# Building Dimensions table for the drivers silver tables

### Getting the batch id as input parameter

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00.Common/01.Environment-config

In [0]:
%run "../00.Common/04.Helper_Notebook_Gold"

In [0]:
target_name = f"{catalog_name}.{gold_schema}.dim_drivers"
drivers_table = f"{catalog_name}.{silver_schema}.drivers"
nationality_table = f"{catalog_name}.{gold_schema}.ref_nationality_region"

### Read the silver tables

In [0]:
#import the sql function and filter the df with the batch id
from pyspark.sql import functions as F
drivers_df = spark.table(drivers_table).filter(F.col("batch_id")==v_batch_id)
nationality_df = spark.table(nationality_table)

### Join the drivers and nationality tables and rename the region column

In [0]:
dim_drivers_df = (
    drivers_df.join(
        nationality_df, drivers_df.nationality == nationality_df.nationality, "left"
        ).select(drivers_df.driver_id, 
                 drivers_df.driver_name,
                 drivers_df.date_of_birth,
                 drivers_df.nationality, 
                 nationality_df.region.alias("nationality_region"))
)

In [0]:
display(dim_drivers_df)

In [0]:
dim_drivers_df.columns

### Write the dataframe into the gold schema

In [0]:
write_to_gold(
    input_df = dim_drivers_df,
    target_table = target_name,
    merge_condition = "t.driver_id=s.driver_id",
    columns_to_update = ['driver_id',
 'driver_name',
 'date_of_birth',
 'nationality',
 'nationality_region']
)

In [0]:
display(spark.table(target_name))